# 07d — Historical Context Enrichment

Primary: exact-year CLCD physical context. Optional: historical OSM/ohsome cross-check.

This stage does not infer occupation, WORK, or OFFICE.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH='eda/07d-historical-context-enrichment'
REPO_DIR=Path('/root/geolife')
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]','rasterio>=1.4,<2'],cwd=REPO_DIR,check=True)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())


In [ ]:
from importlib.util import spec_from_file_location,module_from_spec
import pandas as pd
def load(name,path):
    spec=spec_from_file_location(name,path); m=module_from_spec(spec); sys.modules[name]=m; spec.loader.exec_module(m); return m
s03=load('s03_07d',REPO_DIR/'analysis'/'03a_user_behavior_deep_dive.py')
s07d=load('s07d',REPO_DIR/'analysis'/'07d_historical_context_enrichment.py')
V=Path('/mnt/geolife-data'); C=V/'cache'/'07d_historical_context_enrichment'; C.mkdir(parents=True,exist_ok=True)
xs=sorted(V.glob('**/anchor_observation_dates_private.pkl'))
if not xs: raise FileNotFoundError('Run Stage 07c first: anchor_observation_dates_private.pkl is missing')
anchors=pd.read_pickle(xs[0])

if not {'latitude','longitude'}.issubset(anchors.columns):
    stay_paths=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if not stay_paths: raise FileNotFoundError('stays_baseline_v1.pkl is required to reconstruct private anchor coordinates')
    stays=pd.read_pickle(stay_paths[0])
    resolved=s03.resolve_stay_timezones(stays).dropna(subset=['timezone_id'])
    clustered,_=s03.cluster_behavior_locations(resolved,200.0)
    anchors=s07d.attach_anchor_coordinates_from_clustered_stays(anchors,clustered)
    print('reconstructed private coordinates from frozen 200m clustered stays')

print('anchors:',len(anchors),'users:',anchors['user_id'].astype(str).nunique())
print('coordinate columns present:', {'latitude','longitude'}.issubset(anchors.columns))
display(s07d.build_clcd_sampling_plan(anchors))


## CLCD exact-year physical context

The runner first checks `/mnt/geolife-data/external/clcd/CLCD_v01_<year>.tif`. If absent, it remote-opens the pinned Zenodo Cloud Optimized GeoTIFF.


In [ ]:
LOCAL_CLCD=V/'external'/'clcd'
url_by_year={}
for year in sorted(pd.to_numeric(anchors['median_observation_year']).astype(int).unique()):
    local=LOCAL_CLCD/f'CLCD_v01_{year}_albert.tif'
    if local.exists(): url_by_year[int(year)]=str(local)
print('local CLCD files used:',len(url_by_year))
clcd=s07d.sample_clcd_remote(anchors,url_by_year=url_by_year or None,local_windows=(3,5))
assert len(clcd)==len(anchors), (len(clcd),len(anchors))
print('CLCD POPULATION SUMMARY'); display(s07d.summarize_clcd_population(clcd))
print('CLCD CLASS SUMMARY'); display(s07d.summarize_clcd(clcd))


## Historical OSM cross-check via Modal Secret

This notebook uses the existing Modal secret named `ohsome-api`. The secret must contain `OHSOME_API_KEY`.

The API key is injected only into a small remote Modal function and is never printed or copied into the notebook process.


In [ ]:
import modal

ohsome_app = modal.App('geolife-stage07d-ohsome')
ohsome_secret = modal.Secret.from_name(
    'ohsome-api',
    required_keys=['OHSOME_API_KEY'],
)
ohsome_image = modal.Image.debian_slim(python_version=f'{sys.version_info.major}.{sys.version_info.minor}').pip_install('httpx')

@ohsome_app.function(
    image=ohsome_image,
    secrets=[ohsome_secret],
    serialized=True,
    timeout=300,
)
def _ohsome_post_with_secret(body: dict, api_url: str) -> bytes:
    import os
    import httpx
    response = httpx.post(
        api_url.rstrip('/') + '/extraction/features.parquet',
        json=body,
        headers={'authorization': os.environ['OHSOME_API_KEY']},
        timeout=240,
    )
    response.raise_for_status()
    return response.content

api_url = os.environ.get('OHSOME_API_URL', s07d.OHSOME_API_DEFAULT)
with ohsome_app.run():
    def modal_fetcher(body: dict) -> bytes:
        return _ohsome_post_with_secret.remote(body, api_url)

    ohsome, request_log = s07d.fetch_ohsome_context_with_fetcher(
        anchors,
        fetcher=modal_fetcher,
        cache_dir=C/'ohsome_raw',
        radius_m=100,
        pause_s=0.25,
    )

ohsome_summary = s07d.summarize_ohsome(ohsome)
print('OHSOME SUMMARY')
display(ohsome_summary)


In [ ]:
historical=s07d.join_historical_context(anchors,clcd,ohsome if not ohsome.empty else None)
clcd.to_pickle(C/'clcd_anchor_context_private.pkl')
historical.to_pickle(C/'historical_context_private.pkl')
if not ohsome.empty: ohsome.to_pickle(C/'ohsome_anchor_context_private.pkl')
if not request_log.empty: request_log.to_pickle(C/'ohsome_request_log_private.pkl')
s07d.build_clcd_sampling_plan(anchors).to_csv(C/'clcd_sampling_plan.csv',index=False)
s07d.summarize_clcd_population(clcd).to_csv(C/'clcd_population_summary.csv',index=False)
s07d.summarize_clcd(clcd).to_csv(C/'clcd_class_summary.csv',index=False)
ohsome_summary.to_csv(C/'ohsome_summary.csv',index=False)
print('saved:',C)


## Boundary

- CLCD is physical land cover only; `impervious` is not office/residential/WORK.
- point vs 3×3 / 5×5 agreement is spatial sensitivity, not confidence calibration.
- ohsome is a historical mapping cross-check; missing OSM is not proof of real-world absence.
- BCL POI 2008 remains blocked until exact file access/license/CRS are verified.
